# Import Libraries

In [28]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

## Data Exctraction

In [29]:
df = pd.read_csv("imdb_top_1000.csv")
df

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,https://m.media-amazon.com/images/M/MV5BNGEwMT...,Breakfast at Tiffany's,1961,A,115 min,"Comedy, Drama, Romance",7.6,A young New York socialite becomes interested ...,76.0,Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,166544,NaN
996,https://m.media-amazon.com/images/M/MV5BODk3Yj...,Giant,1956,G,201 min,"Drama, Western",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,34075,NaN
997,https://m.media-amazon.com/images/M/MV5BM2U3Yz...,From Here to Eternity,1953,Passed,118 min,"Drama, Romance, War",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,43374,"30,500,000"
998,https://m.media-amazon.com/images/M/MV5BZTBmMj...,Lifeboat,1944,NaN,97 min,"Drama, War",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,26471,NaN


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Poster_Link    1000 non-null   object 
 1   Series_Title   1000 non-null   object 
 2   Released_Year  1000 non-null   object 
 3   Certificate    899 non-null    object 
 4   Runtime        1000 non-null   object 
 5   Genre          1000 non-null   object 
 6   IMDB_Rating    1000 non-null   float64
 7   Overview       1000 non-null   object 
 8   Meta_score     843 non-null    float64
 9   Director       1000 non-null   object 
 10  Star1          1000 non-null   object 
 11  Star2          1000 non-null   object 
 12  Star3          1000 non-null   object 
 13  Star4          1000 non-null   object 
 14  No_of_Votes    1000 non-null   int64  
 15  Gross          831 non-null    object 
dtypes: float64(2), int64(1), object(13)
memory usage: 125.1+ KB


In [31]:
# Corrected code to select columns

df = df[["Series_Title", "Released_Year", "Genre", "Director", "Star1", "Star2", "Star3"]]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Series_Title   1000 non-null   object
 1   Released_Year  1000 non-null   object
 2   Genre          1000 non-null   object
 3   Director       1000 non-null   object
 4   Star1          1000 non-null   object
 5   Star2          1000 non-null   object
 6   Star3          1000 non-null   object
dtypes: object(7)
memory usage: 54.8+ KB


### Data Organizing

In [32]:

df["combined_features"] = (
    df["Released_Year"].astype(str) + " " +
    df["Genre"].astype(str) + " " + # Note: Re-checking for potential trailing space!
    df["Director"].astype(str) + " " +
    df["Star1"].astype(str) + " " +
    df["Star2"].astype(str) + " " +
    df["Star3"].astype(str)
)
df

C:\Users\duke laptop\AppData\Local\Temp\ipykernel_14980\3473886592.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["combined_features"] = (


,Series_Title,Released_Year,Genre,Director,Star1,Star2,Star3,combined_features
0,The Shawshank Redemption,1994,Drama,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,1994 Drama Frank Darabont Tim Robbins Morgan F...
1,The Godfather,1972,"Crime, Drama",Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,"1972 Crime, Drama Francis Ford Coppola Marlon ..."
2,The Dark Knight,2008,"Action, Crime, Drama",Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,"2008 Action, Crime, Drama Christopher Nolan Ch..."
3,The Godfather: Part II,1974,"Crime, Drama",Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,"1974 Crime, Drama Francis Ford Coppola Al Paci..."
4,12 Angry Men,1957,"Crime, Drama",Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,"1957 Crime, Drama Sidney Lumet Henry Fonda Lee..."
...,...,...,...,...,...,...,...,...
995,Breakfast at Tiffany's,1961,"Comedy, Drama, Romance",Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,"1961 Comedy, Drama, Romance Blake Edwards Audr..."
996,Giant,1956,"Drama, Western",George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,"1956 Drama, Western George Stevens Elizabeth T..."
997,From Here to Eternity,1953,"Drama, Romance, War",Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,"1953 Drama, Romance, War Fred Zinnemann Burt L..."
998,Lifeboat,1944,"Drama, War",Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,"1944 Drama, War Alfred Hitchcock Tallulah Bank..."


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Series_Title       1000 non-null   object
 1   Released_Year      1000 non-null   object
 2   Genre              1000 non-null   object
 3   Director           1000 non-null   object
 4   Star1              1000 non-null   object
 5   Star2              1000 non-null   object
 6   Star3              1000 non-null   object
 7   combined_features  1000 non-null   object
dtypes: object(8)
memory usage: 62.6+ KB


In [34]:
print(type(df))
print(df.head())

<class 'pandas.core.frame.DataFrame'>
               Series_Title Released_Year                 Genre  \
0  The Shawshank Redemption          1994                 Drama   
1             The Godfather          1972          Crime, Drama   
2           The Dark Knight          2008  Action, Crime, Drama   
3    The Godfather: Part II          1974          Crime, Drama   
4              12 Angry Men          1957          Crime, Drama   

               Director           Star1           Star2          Star3  \
0        Frank Darabont     Tim Robbins  Morgan Freeman     Bob Gunton   
1  Francis Ford Coppola   Marlon Brando       Al Pacino     James Caan   
2     Christopher Nolan  Christian Bale    Heath Ledger  Aaron Eckhart   
3  Francis Ford Coppola       Al Pacino  Robert De Niro  Robert Duvall   
4          Sidney Lumet     Henry Fonda     Lee J. Cobb  Martin Balsam   

                                   combined_features  
0  1994 Drama Frank Darabont Tim Robbins Morgan F...  
1  1

### Vectorizing the text as DataFrame

In [35]:

text_data = df
vectorizer = CountVectorizer()
X_sparse = vectorizer.fit_transform(text_data)
text_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Series_Title       1000 non-null   object
 1   Released_Year      1000 non-null   object
 2   Genre              1000 non-null   object
 3   Director           1000 non-null   object
 4   Star1              1000 non-null   object
 5   Star2              1000 non-null   object
 6   Star3              1000 non-null   object
 7   combined_features  1000 non-null   object
dtypes: object(8)
memory usage: 62.6+ KB


In [36]:
print(f"Successfully vectorized data. Resulting shape: {X_sparse.shape}")
print(f"Vocabulary Size: {len(vectorizer.get_feature_names_out())}")


Successfully vectorized data. Resulting shape: (8, 8)
Vocabulary Size: 8


In [37]:
df = df.groupby("Series_Title")["combined_features"].first()
df

Series_Title
(500) Days of Summer           2009 Comedy, Drama, Romance Marc Webb Zooey De...
12 Angry Men                   1957 Crime, Drama Sidney Lumet Henry Fonda Lee...
12 Years a Slave               2013 Biography, Drama, History Steve McQueen C...
1917                           2019 Drama, Thriller, War Sam Mendes Dean-Char...
2001: A Space Odyssey          1968 Adventure, Sci-Fi Stanley Kubrick Keir Du...
                                                     ...                        
Zootopia                       2016 Animation, Adventure, Comedy Byron Howard...
Zulu                           1964 Drama, History, War Cy Endfield Stanley B...
Zwartboek                      2006 Drama, Thriller, War Paul Verhoeven Caric...
À bout de souffle              1960 Crime, Drama Jean-Luc Godard Jean-Paul Be...
Ôkami kodomo no Ame to Yuki    2012 Animation, Drama, Fantasy Mamoru Hosoda A...
Name: combined_features, Length: 999, dtype: object

In [38]:
df.info()

<class 'pandas.core.series.Series'>
Index: 999 entries, (500) Days of Summer to Ôkami kodomo no Ame to Yuki
Series name: combined_features
Non-Null Count  Dtype 
--------------  ----- 
999 non-null    object
dtypes: object(1)
memory usage: 47.9+ KB


### Managing Text as pandas.Series

In [39]:
text_data = text_data.groupby("Series_Title")["combined_features"].first()
text_data

Series_Title
(500) Days of Summer           2009 Comedy, Drama, Romance Marc Webb Zooey De...
12 Angry Men                   1957 Crime, Drama Sidney Lumet Henry Fonda Lee...
12 Years a Slave               2013 Biography, Drama, History Steve McQueen C...
1917                           2019 Drama, Thriller, War Sam Mendes Dean-Char...
2001: A Space Odyssey          1968 Adventure, Sci-Fi Stanley Kubrick Keir Du...
                                                     ...                        
Zootopia                       2016 Animation, Adventure, Comedy Byron Howard...
Zulu                           1964 Drama, History, War Cy Endfield Stanley B...
Zwartboek                      2006 Drama, Thriller, War Paul Verhoeven Caric...
À bout de souffle              1960 Crime, Drama Jean-Luc Godard Jean-Paul Be...
Ôkami kodomo no Ame to Yuki    2012 Animation, Drama, Fantasy Mamoru Hosoda A...
Name: combined_features, Length: 999, dtype: object

In [40]:
X_sparse = vectorizer.fit_transform(text_data)

In [41]:
print(X_sparse.shape)  # ابعاد ماتریس ویژگی‌ها
print(vectorizer.get_feature_names_out()[:15])  # چند تا ویژگی اول

(999, 3602)
['1920' '1921' '1922' '1924' '1925' '1926' '1927' '1928' '1930' '1931'
 '1932' '1933' '1934' '1935' '1936']


#### Finf Similarities between Titles shown as index

In [42]:
cosine_sim = cosine_similarity(X_sparse, X_sparse)
cosine_sim

array([[1.        , 0.0836242 , 0.07161149, ..., 0.07692308, 0.06362848,
        0.08006408],
       [0.0836242 , 1.        , 0.07784989, ..., 0.0836242 , 0.13834289,
        0.08703883],
       [0.07161149, 0.07784989, 1.        , ..., 0.07161149, 0.05923489,
        0.0745356 ],
       ...,
       [0.07692308, 0.0836242 , 0.07161149, ..., 1.        , 0.12725695,
        0.08006408],
       [0.06362848, 0.13834289, 0.05923489, ..., 0.12725695, 1.        ,
        0.06622662],
       [0.08006408, 0.08703883, 0.0745356 , ..., 0.08006408, 0.06622662,
        1.        ]], shape=(999, 999))

#### Find Index regarding Title from the Prime Table

In [43]:
indices = pd.Series(range(len(text_data)), index=text_data.index)
indices

Series_Title
(500) Days of Summer             0
12 Angry Men                     1
12 Years a Slave                 2
1917                             3
2001: A Space Odyssey            4
                              ... 
Zootopia                       994
Zulu                           995
Zwartboek                      996
À bout de souffle              997
Ôkami kodomo no Ame to Yuki    998
Length: 999, dtype: int64

#### Test for accredited data

In [44]:
movie_features = text_data.loc["Inception"]
movie_features

'2010 Action, Adventure, Sci-Fi Christopher Nolan Leonardo DiCaprio Joseph Gordon-Levitt Elliot Page'

#### Test for accredited index

In [45]:
movie_index_in_matrix = indices.loc["Inception"]
movie_index_in_matrix

np.int64(374)

#### Raw similarity detection

In [46]:
sim_scores = list(enumerate(cosine_sim[374]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
print(type(sim_scores))

<class 'list'>


In [47]:
indices.loc["Fight Club"]

np.int64(274)

#### Function for recommendation

In [48]:
def recommend(title, n=5):
    # بررسی اینکه فیلم وجود دارد یا نه
    if title not in indices.index:
        return f"❌ Movie '{title}' is not available."

    idx = indices[title]

    sim_scores = list(enumerate(cosine_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    top_results = sim_scores[1:n + 1]

    similar_movies = pd.DataFrame({
        "Index": [i[0] for i in top_results],  # اندیس فیلم در ماتریس X_sparse
        "Title": text_data.index[[i[0] for i in top_results]],  # نام فیلم
        "Similarity": [i[1] for i in top_results]  # امتیاز شباهت
    })

    similar_movies = similar_movies.sort_values(by="Similarity", ascending=False).reset_index(drop=True)

    return similar_movies



In [49]:
recommend("Aliens", n=10)

,Index,Title,Similarity
0,887,The Terminator,0.560449
1,72,Avatar,0.480384
2,982,X: First Class,0.461538
3,737,Terminator 2: Judgment Day,0.400320
4,499,Mad Max 2,0.384615
5,39,Alien,0.320256
6,714,Star Trek,0.320256
7,716,Star Trek Into Darkness,0.320256
8,720,Star Wars: Episode VII - The Force Awakens,0.320256
9,83,Back to the Future,0.307692


In [50]:
recommend("Fight Club", n=10)

,Index,Title,Similarity
0,663,Se7en,0.456435
1,765,The Curious Case of Benjamin Button,0.456435
2,50,American History X,0.365148
3,6,25th Hour,0.286039
4,109,Birdman or (The Unexpected Virtue of Ignorance),0.286039
5,582,Once Upon a Time... in Hollywood,0.286039
6,880,The Social Network,0.286039
7,377,Inglourious Basterds,0.273861
8,533,Moneyball,0.273861
9,617,Primal Fear,0.273861
